# Nome do Aluno: Milton Salgado Leandro
# Instituição de Ensino: Universidade Federal do Rio de Janeiro

# Silver_to_Gold — Modelagem Dimensional e Analytics

**Task no Workflow:** `to_Gold`  
**Dependência:** executa somente após `to_Silver`.

**Objetivo:** construir o Star Schema da camada Gold e executar as consultas analíticas sobre dados já tratados.

- **Fato:** `fact_movies_performance` — um registro por filme lançado.
- **Dimensões:** `dim_movies`, `dim_genres`, `dim_people`, `dim_companies`, `dim_reviews`.
- **Pontes:** `bridge_movie_genre`, `bridge_movie_person`, `bridge_movie_company`.

As chaves substitutas usam SHA-256 reduzido a `BIGINT` e são verificadas contra colisões. As tabelas Gold usam Delta/Overwrite, permitindo reconstrução recorrente pelo Workflow.

**Limitação de origem:** a identidade usa `id_filme` (TMDB). IDs TMDB distintos associados ao mesmo IMDb não são mesclados sem uma regra canônica explícita.


In [0]:
from pyspark.sql import functions as F

CATALOGO = "workspace"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Quality Gate de entrada: a Gold só inicia
# quando todas as fontes Silver estão disponíveis.
fontes_silver = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

for tabela in fontes_silver:
    nome = f"silver.{tabela}"

    if not spark.catalog.tableExists(nome):
        raise RuntimeError(
            f"Fonte Silver ausente: {nome}"
        )

    if spark.table(nome).limit(1).count() == 0:
        raise RuntimeError(
            f"Fonte Silver vazia: {nome}"
        )

print("Fontes Silver validadas. Iniciando Gold.")


# Salvar tabelas Gold em Delta.
def salvar_gold(df, nome):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"gold.{nome}")
    )


# SHA-256 reduzido a 60 bits:
# chave BIGINT estável e distribuída.
def adicionar_sk(df, nome, colunas):

    chave = F.to_json(
        F.struct(
            *[F.col(c) for c in colunas]
        ),
        {"ignoreNullFields": "false"}
    )

    hash_256 = F.sha2(chave, 256)

    sk = F.conv(
        F.substring(hash_256, 1, 15),
        16,
        10
    ).cast("bigint")

    return df.withColumn(nome, sk)


Fontes Silver validadas. Iniciando Gold.


# Entrega 1.1 — `gold.dim_movies`

**Origem:** `silver.tb_info_filmes`  
**Destino:** `gold.dim_movies`

**Grão:** um registro por `id_filme` (TMDB), com `sk_movie_id` BIGINT, `id_filme` STRING, `titulo` STRING, `data_lancamento` DATE, `ano_lancamento` INT, `duracao_minutos` INT, `idioma_original` STRING, `status_filme` STRING e `sinopse` STRING. A chave substituta é gerada com hash do ID natural.


In [0]:
dim_movies = (
    spark.table("silver.tb_info_filmes")
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
    .dropDuplicates(["id_filme"])
)

dim_movies = adicionar_sk(
    dim_movies,
    "sk_movie_id",
    ["id_filme"]
)

dim_movies = dim_movies.select(
    "sk_movie_id",
    "id_filme",
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse"
)

salvar_gold(dim_movies, "dim_movies")

display(dim_movies.limit(20))

sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
674093378270249182,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
490559522768335606,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,null
125556648400748868,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
635124435733081277,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
1091491847064123078,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
1027509291634534039,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null
420842760481078661,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
1140804060083952365,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
257234108079517852,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
547795061967628164,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."


# Entrega 1.2 — `gold.dim_genres`

**Origem:** `silver.tb_generos`  
**Destino:** `gold.dim_genres`

**Regra:** catálogo único de gêneros cinematográficos, com `sk_genre_id` BIGINT e `nome_genero` STRING. Os relacionamentos com filmes serão mantidos em `bridge_movie_genre`.


In [0]:

dim_genres = (
    spark.table("silver.tb_generos")
    .select("nome_genero")
    .distinct()
)

dim_genres = adicionar_sk(
    dim_genres,
    "sk_genre_id",
    ["nome_genero"]
)

dim_genres = dim_genres.select(
    "sk_genre_id",
    "nome_genero"
)

salvar_gold(dim_genres, "dim_genres")

display(dim_genres.limit(20))

sk_genre_id,nome_genero
130476127076471178,Mystery
307252778967214984,Animation
739944298357213727,Science Fiction
692139654811968036,Action
973237878082420663,Family
245933323459137096,Romance
774000812449878990,History
680008854125927550,Comedy
297563567325695942,Crime
1077741394699031028,Thriller


# Entrega 1.3 — `gold.dim_people`

**Origem:** `silver.tb_pessoas_empresas`  
**Destino:** `gold.dim_people`

**Regra:** manter apenas `Ator`, `Diretor` e `Roteirista`; deduplicar `(nome_pessoa, tipo_pessoa)` e gerar `sk_person_id` BIGINT. A mesma pessoa pode ocupar mais de um papel e, portanto, ter registros distintos por função.


In [0]:
entidades = spark.table(
    "silver.tb_pessoas_empresas"
)

dim_people = (
    entidades
    .filter(
        F.col("tipo_entidade").isin(
            "Ator", "Diretor", "Roteirista"
        )
    )
    .select(
        F.col("nome_entidade").alias(
            "nome_pessoa"
        ),
        F.col("tipo_entidade").alias(
            "tipo_pessoa"
        )
    )
    .distinct()
)

dim_people = adicionar_sk(
    dim_people,
    "sk_person_id",
    ["nome_pessoa", "tipo_pessoa"]
)

dim_people = dim_people.select(
    "sk_person_id",
    "nome_pessoa",
    "tipo_pessoa"
)

salvar_gold(dim_people, "dim_people")

display(dim_people.limit(20))

sk_person_id,nome_pessoa,tipo_pessoa
927400022984467288,Josh Brolin,Ator
59579642110144035,Zazie Beetz,Ator
983519080529001837,Alfonso Arau,Ator
9847650205043209,Izzie Coffey,Ator
145117145699627626,J.k. Simmons,Ator
466042865822058736,Brie Larson,Ator
406251844824884127,Domhnall Gleeson,Ator
713770815649249239,Gal Gadot,Ator
487046699240141848,Timothy Olyphant,Ator
556430119926402912,Troy Polamalu,Ator


# Entrega 1.4 — `gold.dim_companies`

**Origem:** `silver.tb_pessoas_empresas`  
**Destino:** `gold.dim_companies`

**Regra:** manter apenas `Produtora`, deduplicar `nome_produtora` e gerar `sk_company_id` BIGINT. As associações com os filmes ficam em `bridge_movie_company`.


In [0]:
dim_companies = (
    entidades
    .filter(
        F.col("tipo_entidade") == "Produtora"
    )
    .select(
        F.col("nome_entidade").alias(
            "nome_produtora"
        )
    )
    .distinct()
)

dim_companies = adicionar_sk(
    dim_companies,
    "sk_company_id",
    ["nome_produtora"]
)

dim_companies = dim_companies.select(
    "sk_company_id",
    "nome_produtora"
)

salvar_gold(
    dim_companies, "dim_companies"
)

display(dim_companies.limit(20))

sk_company_id,nome_produtora
945179146052904033,Village Roadshow Pictures
166022788282063442,The Safran Company
1113577835230290811,Scott Free Productions
240800659648875996,Sunswept Entertainment
1022599037442685616,Jerry Bruckheimer Films
1152707153619279604,Thunder Road
125226363117359822,One Race
141812130243291889,Malpaso Productions
128715049913272621,Walt Disney Pictures
937769047982469166,Keep Your Head Productions


# Entrega 1.5 — `gold.dim_reviews`

**Origem:** `silver.tb_avaliacoes_usuarios`  
**Destino:** `gold.dim_reviews`

**Grão:** uma linha por filme com avaliações. Relacionar por `sk_movie_id`, gerar `sk_review_id` BIGINT, contar avaliações (`qtd_avaliacoes_usuarios` INT) e calcular a média válida (`nota_media_usuarios` DOUBLE, arredondada a duas casas). Notas `NULL` não entram na média.


In [0]:
reviews = spark.table(
    "silver.tb_avaliacoes_usuarios"
)

dim_reviews = (
    reviews
    .join(
        dim_movies.select(
            "id_filme", "sk_movie_id"
        ),
        "id_filme",
        "inner"
    )
    .groupBy("sk_movie_id")
    .agg(
        F.count("*").cast("int").alias(
            "qtd_avaliacoes_usuarios"
        ),
        F.round(
            F.avg("nota_usuario"), 2
        ).cast("double").alias(
            "nota_media_usuarios"
        )
    )
)

dim_reviews = adicionar_sk(
    dim_reviews,
    "sk_review_id",
    ["sk_movie_id"]
)

dim_reviews = dim_reviews.select(
    "sk_review_id",
    "sk_movie_id",
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios"
)

salvar_gold(dim_reviews, "dim_reviews")

display(dim_reviews.limit(20))

sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
227060811317043464,1102010304275940186,1,4.4
836316766664202068,638042426832549347,1,5.4
425113553158269389,9459126083148427,1,2.6
194665062695734333,872235866910752570,1,7.4
706781354794115554,391884821741699612,2,5.95
140160242512685346,1133095770151580982,1,2.4
36323779189073820,789868442715143393,1,6.4
657845768769117920,69725705931376654,1,7.7
137948971947596414,1144935863627064299,1,6.5
818470827578119780,459053607491099500,1,2.4


# Entrega 1.6 — Tabelas-Ponte

Criar as associações deduplicadas `(sk_movie_id, sk_genre_id)`, `(sk_movie_id, sk_person_id)` e `(sk_movie_id, sk_company_id)` nas tabelas `bridge_movie_genre`, `bridge_movie_person` e `bridge_movie_company`. As pontes representam muitos-para-muitos sem multiplicar o grão da fato.


In [0]:
# Associação filme–gênero.

bridge_genre = (
    spark.table("silver.tb_generos")
    .join(
        dim_movies.select(
            "id_filme", "sk_movie_id"
        ),
        "id_filme",
        "inner"
    )
    .join(
        dim_genres,
        "nome_genero",
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_genre_id"
    )
    .distinct()
)

salvar_gold(
    bridge_genre,
    "bridge_movie_genre"
)


# Associação filme–pessoa e função.

pessoas = (
    entidades
    .filter(
        F.col("tipo_entidade").isin(
            "Ator", "Diretor", "Roteirista"
        )
    )
    .select(
        "id_filme",
        F.col("nome_entidade").alias(
            "nome_pessoa"
        ),
        F.col("tipo_entidade").alias(
            "tipo_pessoa"
        )
    )
)

bridge_person = (
    pessoas
    .join(
        dim_movies.select(
            "id_filme", "sk_movie_id"
        ),
        "id_filme",
        "inner"
    )
    .join(
        dim_people,
        ["nome_pessoa", "tipo_pessoa"],
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_person_id"
    )
    .distinct()
)

salvar_gold(
    bridge_person,
    "bridge_movie_person"
)


# Associação filme–produtora.

produtoras = (
    entidades
    .filter(
        F.col("tipo_entidade") == "Produtora"
    )
    .select(
        "id_filme",
        F.col("nome_entidade").alias(
            "nome_produtora"
        )
    )
)

bridge_company = (
    produtoras
    .join(
        dim_movies.select(
            "id_filme", "sk_movie_id"
        ),
        "id_filme",
        "inner"
    )
    .join(
        dim_companies,
        "nome_produtora",
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_company_id"
    )
    .distinct()
)

salvar_gold(
    bridge_company,
    "bridge_movie_company"
)

display(bridge_genre.limit(10))
display(bridge_person.limit(10))
display(bridge_company.limit(10))


sk_movie_id,sk_genre_id
37719249003732147,245933323459137096
859855347707673719,402820578603923989
445493506779896432,402820578603923989
1103918115714471645,973237878082420663
166176094163131388,297563567325695942
642438321518062448,1109037792083730482
1148253530283772545,1077741394699031028
649333285727970813,245933323459137096
247558241126174014,739944298357213727
1101223355349139740,692139654811968036


sk_movie_id,sk_person_id
185120466166913683,382771375050639051
644033930310968550,903510417917951622
408575226707768312,215214970515738832
739691048808463197,453191141020304159
1110328831940935296,309572119423018178
619533171303369945,893869771352042173
765469294061280785,916681123032825080
16028376520732238,930205917301551313
363193378672518829,827607764077434145
264428762720174378,217422450121446726


sk_movie_id,sk_company_id
573214133263521457,954676689326010883
543382237750669143,858359565942139664
143346725008628765,630359925542344296
672594284000975387,1139697361646132500
899594347898611703,107886879358801038
287570360019239279,860099557061957440
545855581787479487,351056052118982400
26573323325251761,388341134797500693
282829557157343828,792813448347121025
580618110418204518,841495653112163773


# Entrega 1.7 — `gold.fact_movies_performance`

**Origens:** `silver.tb_financeiro_filmes`, `silver.tb_metricas_engajamento`, `gold.dim_movies`  
**Destino:** `gold.fact_movies_performance`

**Grão:** um registro por filme efetivamente lançado, com `sk_movie_id` BIGINT como FK. Consolidar `orcamento_usd`, `receita_usd`, `lucro_usd`, `orcamento_brl`, `receita_brl`, `lucro_brl` em `DECIMAL(18,2)`; `popularidade`, `nota_media_tmdb`, `nota_media_imdb` em `DOUBLE`; `qtd_votos_tmdb`, `qtd_votos_imdb` em `INT`.

**Regra:** usar joins por filme com tabelas Silver já deduplicadas; excluir filmes não lançados e datas futuras. Relações com atores, gêneros e empresas são consultadas pelas pontes, não diretamente na construção da fato.


In [0]:
financeiro = spark.table(
    "silver.tb_financeiro_filmes"
)

metricas = spark.table(
    "silver.tb_metricas_engajamento"
)

# Selecionar filmes efetivamente lançados.
filmes_lancados = dim_movies.filter(
    (F.col("status_filme") == "Lançado")
    & (
        F.col("data_lancamento")
        <= F.current_date()
    )
)

# Consolidar as métricas por filme.
fato = (
    filmes_lancados
    .join(
        financeiro,
        "id_filme",
        "left"
    )
    .join(
        metricas,
        "id_filme",
        "left"
    )
)

# Selecionar as colunas solicitadas.
fato = fato.select(
    "sk_movie_id",

    F.col("orcamento_usd").cast(
        "decimal(18,2)"
    ),
    F.col("receita_usd").cast(
        "decimal(18,2)"
    ),
    F.col("lucro_usd").cast(
        "decimal(18,2)"
    ),

    F.col("orcamento_brl").cast(
        "decimal(18,2)"
    ),
    F.col("receita_brl").cast(
        "decimal(18,2)"
    ),
    F.col("lucro_brl").cast(
        "decimal(18,2)"
    ),

    F.col("popularidade").cast("double"),
    F.col("nota_media_tmdb").cast("double"),
    F.col("qtd_votos_tmdb").cast("int"),
    F.col("nota_media_imdb").cast("double"),
    F.col("qtd_votos_imdb").cast("int")
)

salvar_gold(
    fato,
    "fact_movies_performance"
)

display(fato.limit(20))

sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
490559522768335606,null,null,null,null,null,null,0.6,0.0,0,null,40
1082895680931110764,null,null,null,null,null,null,0.914,10.0,1,7.5,93
445201812111925314,6000000.00,null,null,29957400.00,null,null,6.518,6.833,33,6.5,1421
732753286478169549,null,null,null,null,null,null,1.795,0.0,0,null,635
915368750126708149,null,null,null,null,null,null,2.542,5.7,3,null,143
563155042298825076,null,null,null,null,null,null,1.875,9.0,2,7.4,null
867424213336721845,null,null,null,null,null,null,0.6,0.0,0,6.3,9
708334370225884460,null,null,null,null,null,null,2.239,0.0,0,5.5,104
1139486995456693193,null,null,null,null,null,null,311.77,8.011,179,null,26954
1118445402654359360,null,null,null,null,null,null,20.777,7.0,35,6.8,3195


# Validação do Star Schema

Confirmar a existência das nove tabelas Gold, unicidade do grão da fato, integridade das três pontes e ausência de colisões ou nulos nas chaves substitutas das cinco dimensões.


In [0]:
tabelas_gold = [
    "fact_movies_performance",
    "dim_movies",
    "dim_genres",
    "dim_people",
    "dim_companies",
    "dim_reviews",
    "bridge_movie_genre",
    "bridge_movie_person",
    "bridge_movie_company"
]

for tabela in tabelas_gold:

    df = spark.table(f"gold.{tabela}")

    print(
        f"{tabela}: {df.count()} registros"
    )


# Um registro por filme na fato.
fato = spark.table(
    "gold.fact_movies_performance"
)

total = fato.count()

unicos = fato.select(
    "sk_movie_id"
).distinct().count()

assert total == unicos, (
    "A tabela fato contém filmes duplicados!"
)


# Integridade das FKs da fato.
fks_invalidas = (
    fato
    .join(
        dim_movies.select("sk_movie_id"),
        "sk_movie_id",
        "left_anti"
    )
    .count()
)

assert fks_invalidas == 0

print("Validação básica da fato concluída!")


# Unicidade e integridade das pontes.
pontes = [
    (
        "bridge_movie_genre",
        "sk_genre_id",
        "dim_genres"
    ),
    (
        "bridge_movie_person",
        "sk_person_id",
        "dim_people"
    ),
    (
        "bridge_movie_company",
        "sk_company_id",
        "dim_companies"
    )
]

for nome, coluna, dimensao in pontes:

    ponte = spark.table(f"gold.{nome}")
    dim = spark.table(f"gold.{dimensao}")

    assert ponte.count() == (
        ponte.select(
            "sk_movie_id", coluna
        ).distinct().count()
    )

    assert ponte.join(
        dim_movies.select("sk_movie_id"),
        "sk_movie_id",
        "left_anti"
    ).count() == 0

    assert ponte.join(
        dim.select(coluna),
        coluna,
        "left_anti"
    ).count() == 0

print("Pontes validadas!")

# Verificar unicidade e ausência de colisões nas SKs.
dimensoes = {
    "dim_movies": "sk_movie_id",
    "dim_genres": "sk_genre_id",
    "dim_people": "sk_person_id",
    "dim_companies": "sk_company_id",
    "dim_reviews": "sk_review_id"
}

for tabela, chave in dimensoes.items():
    df_dim = spark.table(f"gold.{tabela}")
    assert df_dim.count() == df_dim.select(chave).distinct().count(), (
        f"Chave não única: gold.{tabela}"
    )
    assert df_dim.filter(F.col(chave).isNull()).count() == 0, (
        f"Chave nula: gold.{tabela}"
    )

print("Chaves substitutas validadas!")


fact_movies_performance: 96460 registros
dim_movies: 97879 registros
dim_genres: 19 registros
dim_people: 420684 registros
dim_companies: 45445 registros
dim_reviews: 27303 registros
bridge_movie_genre: 140243 registros
bridge_movie_person: 780086 registros
bridge_movie_company: 118340 registros
Validação básica da fato concluída!
Pontes validadas!
Chaves substitutas validadas!


# 3 — Orquestração e Otimização

Este notebook foi preparado para a tarefa `to_Gold` do Databricks Workflow.

A dependência esperada do Job é:

`to_Bronze` → `to_Silver` → `to_Gold`

O agendamento e as dependências são configurados no Databricks Workflows e exportados posteriormente no arquivo `job.yaml`. Os Quality Gates dos notebooks fazem a tarefa falhar quando uma camada de origem está ausente ou inválida, impedindo a continuação indevida do pipeline.


# 4.1 — Receita Total dos Filmes

**Pergunta:** Qual é a receita total (R$) dos filmes da base?

Somar `receita_brl` na fato, sem joins com tabelas-ponte para evitar duplicação. A fato considera filmes lançados; receitas ausentes são ignoradas pelo `SUM`.


In [0]:
display(spark.sql("""
    SELECT
        ROUND(
            SUM(receita_brl), 2
        ) AS receita_total_brl

    FROM gold.fact_movies_performance
"""))

receita_total_brl
803261918455.39


# 4.2 — Top 5 por Popularidade

**Pergunta:** Quais são os cinco filmes com maior popularidade?

Relacionar fato e `dim_movies` por `sk_movie_id` e ordenar `popularidade` decrescente, deixando `NULL` ao final. Exibir título e popularidade.


In [0]:
display(spark.sql("""
    SELECT
        m.titulo,
        f.popularidade

    FROM gold.fact_movies_performance f

    JOIN gold.dim_movies m
        ON f.sk_movie_id = m.sk_movie_id

    ORDER BY
        f.popularidade DESC NULLS LAST

    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


# 4.3 — Filmes por Gênero

**Pergunta:** Quantos filmes cada gênero possui?

Relacionar `bridge_movie_genre` à `dim_genres`, contar IDs de filmes distintos e ordenar do maior para o menor volume. O catálogo de gêneros pode incluir filmes ainda não lançados.


In [0]:
display(spark.sql("""
    SELECT
        g.nome_genero,

        COUNT(
            DISTINCT b.sk_movie_id
        ) AS qtd_filmes

    FROM gold.bridge_movie_genre b

    JOIN gold.dim_genres g
        ON b.sk_genre_id = g.sk_genre_id

    GROUP BY
        g.nome_genero

    ORDER BY
        qtd_filmes DESC
"""))

nome_genero,qtd_filmes
Drama,32251
Documentary,18983
Comedy,18588
Thriller,10266
Horror,9716
Romance,7628
Action,6035
Crime,4733
Animation,4459
TV Movie,4071


# 4.4 — Top 10 de Receita

**Pergunta:** Quais são os dez filmes de maior receita, com valores USD/BRL e posição?

Aplicar `RANK()` por `receita_usd` decrescente e relacionar a fato à dimensão de filmes. Empates recebem a mesma posição.


In [0]:
display(spark.sql("""
    WITH ranking AS (
        SELECT
            m.titulo,
            f.receita_usd,
            f.receita_brl,

            RANK() OVER (
                ORDER BY
                    f.receita_usd DESC
                    NULLS LAST
            ) AS posicao

        FROM gold.fact_movies_performance f

        JOIN gold.dim_movies m
            ON f.sk_movie_id = m.sk_movie_id

        WHERE f.receita_usd IS NOT NULL
    )

    SELECT *
    FROM ranking

    ORDER BY
        posicao,
        titulo

    LIMIT 10
"""))

titulo,receita_usd,receita_brl,posicao
Avengers: Endgame,2800000000.00,13980120000.00,1
Avatar: The Way of Water,2320250281.00,11584777628.00,2
AVENGERS: INFINITY WAR,2052415039.00,10247503048.22,3
spider-man: no way home,1921847111.00,9595590440.51,4
The Lion King,1663075401.00,8303569169.65,5
Top Gun: Maverick,1488732821.00,7433094101.97,6
Barbie,1428545028.00,7132582470.30,7
The Super Mario Bros. Movie,1355725263.00,6769000665.63,8
Black Panther,1349926083.00,6740045939.81,9
Star Wars: The Last Jedi,1332698830.00,6654031988.31,10


# 4.5 — Ator com Mais Participações (2 anos)

**Pergunta:** Qual ator teve mais participações em filmes lançados nos últimos dois anos?

Usar como limite superior a **data válida mais recente de lançamento na própria base** (excluir futuros/não lançados). Contar filmes distintos por ator nos 24 meses anteriores, via `bridge_movie_person`. A identidade por ID TMDB pode superestimar filmes quando diferentes IDs compartilham um IMDb.


In [0]:
display(spark.sql("""
    WITH limite AS (
        SELECT
            MAX(data_lancamento)
                AS data_maxima

        FROM gold.dim_movies

        WHERE status_filme = 'Lançado'
          AND data_lancamento
              <= CURRENT_DATE()
    )

    SELECT
        p.nome_pessoa,

        COUNT(
            DISTINCT m.sk_movie_id
        ) AS participacoes

    FROM gold.dim_movies m

    JOIN gold.bridge_movie_person b
        ON m.sk_movie_id = b.sk_movie_id

    JOIN gold.dim_people p
        ON b.sk_person_id = p.sk_person_id

    CROSS JOIN limite l

    WHERE p.tipo_pessoa = 'Ator'

      AND m.status_filme = 'Lançado'

      AND m.data_lancamento BETWEEN
          ADD_MONTHS(l.data_maxima, -24)
          AND l.data_maxima

    GROUP BY
        p.nome_pessoa

    ORDER BY
        participacoes DESC

    LIMIT 1
"""))

nome_pessoa,participacoes
Kevin Hart,64


### Limitação dos Dados

Notei que contagem de participações considera filmes distintos pelo identificador `id_filme` (TMDB), conforme a modelagem adotada.

Entretanto, a base original contém diferentes identificadores TMDB associados ao mesmo identificador IMDb. Assim, uma mesma obra pode ser contabilizada mais de uma vez.

Por esse motivo, o resultado representa a quantidade de participações em registros distintos da base, não necessariamente o número de obras cinematográficas únicas.

# 4.6 — Produtora com Maior Lucro (5 anos)

**Pergunta:** Qual produtora teve maior lucro nos últimos cinco anos?

Usar como limite a data de lançamento válida mais recente da base e agregar `lucro_brl` dos filmes lançados nos 60 meses anteriores. Como a fonte não informa participações financeiras, atribuir o **lucro integral do filme a cada produtora associada**. Esses totais não devem ser somados entre empresas.


In [0]:
display(spark.sql("""
    WITH limite AS (
        SELECT
            MAX(data_lancamento)
                AS data_maxima

        FROM gold.dim_movies

        WHERE status_filme = 'Lançado'
          AND data_lancamento
              <= CURRENT_DATE()
    )

    SELECT
        c.nome_produtora,

        ROUND(
            SUM(f.lucro_brl), 2
        ) AS lucro_total_brl

    FROM gold.fact_movies_performance f

    JOIN gold.dim_movies m
        ON f.sk_movie_id = m.sk_movie_id

    JOIN gold.bridge_movie_company b
        ON m.sk_movie_id = b.sk_movie_id

    JOIN gold.dim_companies c
        ON b.sk_company_id = c.sk_company_id

    CROSS JOIN limite l

    WHERE m.data_lancamento BETWEEN
        ADD_MONTHS(l.data_maxima, -60)
        AND l.data_maxima

    GROUP BY
        c.nome_produtora

    ORDER BY
        lucro_total_brl DESC NULLS LAST

    LIMIT 1
"""))

nome_produtora,lucro_total_brl
Universal Pictures,28820664854.27


# 5 — Validação Final e Entrega

**Entregáveis da Atividade 3:**
- `Landing_to_Bronze.ipynb`
- `Bronze_to_Silver.ipynb`
- `Silver_to_Gold.ipynb`
- `job.yaml` exportado do Databricks Workflow
- print da execução bem-sucedida mostrando as dependências entre as tarefas
- repositório GitHub público

**Workflow esperado:** `to_Bronze` → `to_Silver` → `to_Gold`, com agendamento configurado no Databricks.


## 5.1 — Conferência das Tabelas

Listar as tabelas Bronze, Silver e Gold para confirmar a presença de todos os objetos antes de exportar os notebooks.


In [0]:
# Conferir todas as tabelas criadas
# nas três camadas da arquitetura.

for camada in ["bronze", "silver", "gold"]:

    tabelas = spark.sql(
        f"SHOW TABLES IN {camada}"
    )

    print(f"\nCAMADA {camada.upper()}")

    display(tabelas)

# Quality Gate final do Workflow.
tabelas_gold_esperadas = [
    "fact_movies_performance",
    "dim_movies",
    "dim_genres",
    "dim_people",
    "dim_companies",
    "dim_reviews",
    "bridge_movie_genre",
    "bridge_movie_person",
    "bridge_movie_company"
]

for tabela in tabelas_gold_esperadas:
    nome = f"gold.{tabela}"

    if not spark.catalog.tableExists(nome):
        raise RuntimeError(
            f"Tabela Gold ausente: {nome}"
        )

    if spark.table(nome).limit(1).count() == 0:
        raise RuntimeError(
            f"Tabela Gold vazia: {nome}"
        )

print("\nGOLD VALIDADA COM SUCESSO!")



CAMADA BRONZE


database,tableName,isTemporary
bronze,tb_cotacao_dolar,false
bronze,tb_credits_and_tags,false
bronze,tb_ingestion_control,false
bronze,tb_movies_financials,false
bronze,tb_movies_info,false
bronze,tb_movies_metrics,false
bronze,tb_movies_reviews,false



CAMADA SILVER


database,tableName,isTemporary
silver,tb_avaliacoes_usuarios,false
silver,tb_cotacao_dolar,false
silver,tb_financeiro_filmes,false
silver,tb_generos,false
silver,tb_info_filmes,false
silver,tb_metricas_engajamento,false
silver,tb_pessoas_empresas,false



CAMADA GOLD


database,tableName,isTemporary
gold,bridge_movie_company,false
gold,bridge_movie_genre,false
gold,bridge_movie_person,false
gold,dim_companies,false
gold,dim_genres,false
gold,dim_movies,false
gold,dim_people,false
gold,dim_reviews,false
gold,fact_movies_performance,false



GOLD VALIDADA COM SUCESSO!
